# TRACE — Feature Engineering

## Purpose
This notebook builds and validates the engineered features used by TRACE for fraud detection.

The feature groups demonstrated here include transaction, missingness, temporal, entity, and relationship features.

> **Important Note:** Feature engineering is kept explicit so that the same feature definitions can be reproduced during inference.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from features.feature_engineering import FeatureEngineer
from features.feature_validation import FeatureValidator

## 1. Data Loading & Feature Base

In [2]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "extracted_data"
train_transaction = pd.read_csv(DATA_PATH / "train_transaction.csv")
train_identity = pd.read_csv(DATA_PATH / "train_identity.csv")
print("Train Transaction:", train_transaction.shape)
print("Train Identity:", train_identity.shape)

Train Transaction: (590540, 394)
Train Identity: (144233, 41)


In [3]:
feature_base = train_transaction.merge(train_identity[["TransactionID", "DeviceInfo", "DeviceType"]],on="TransactionID",how="left")
print("Transaction shape:", train_transaction.shape)
print("Feature base shape:", feature_base.shape)

Transaction shape: (590540, 394)
Feature base shape: (590540, 396)


In [4]:
feature_base[["TransactionID", "DeviceInfo", "DeviceType"]].head()

,TransactionID,DeviceInfo,DeviceType
0,2987000,NaN,NaN
1,2987001,NaN,NaN
2,2987002,NaN,NaN
3,2987003,NaN,NaN
4,2987004,SAMSUNG SM-G892A Build/NRD90M,mobile


## 2. Transaction Features

In [5]:
feature_engineer = FeatureEngineer()
transaction_features = feature_engineer.create_transaction_features(feature_base)

In [6]:
transaction_features[["TransactionAmt", "log_transaction_amount"]].head()

,TransactionAmt,log_transaction_amount
0,68.5,4.241327
1,29.0,3.401197
2,59.0,4.094345
3,50.0,3.931826
4,50.0,3.931826


In [7]:
transaction_features[["TransactionAmt", "log_transaction_amount"]].describe()

,TransactionAmt,log_transaction_amount
count,590540.000000,590540.000000
mean,135.027176,4.382960
std,239.162522,0.937183
min,0.251000,0.223943
25%,43.321000,3.791459
50%,68.769000,4.245190
75%,125.000000,4.836282
max,31937.391000,10.371564


## 3. Missingness Features

In [8]:
missingness_columns = ["addr1","addr2","D7","D12","D13","D14","DeviceInfo"]
transaction_features = feature_engineer.create_missingness_features(transaction_features,missingness_columns)

In [9]:
print(hasattr(FeatureEngineer, "create_missingness_features"))

True


In [10]:
transaction_features[missingness_columns + [f"{col}_missing" for col in missingness_columns]].head()

,addr1,addr2,D7,D12,D13,D14,DeviceInfo,addr1_missing,addr2_missing,D7_missing,D12_missing,D13_missing,D14_missing,DeviceInfo_missing
0,315.0,87.0,NaN,NaN,NaN,NaN,NaN,0,0,1,1,1,1,1
1,325.0,87.0,NaN,NaN,NaN,NaN,NaN,0,0,1,1,1,1,1
2,330.0,87.0,NaN,NaN,NaN,NaN,NaN,0,0,1,1,1,1,1
3,476.0,87.0,NaN,NaN,NaN,NaN,NaN,0,0,1,1,1,1,1
4,420.0,87.0,NaN,NaN,NaN,NaN,SAMSUNG SM-G892A Build/NRD90M,0,0,1,1,1,1,0


In [11]:
transaction_features[[f"{col}_missing" for col in missingness_columns]].sum()

addr1_missing          65706
addr2_missing          65706
D7_missing            551623
D12_missing           525823
D13_missing           528588
D14_missing           528353
DeviceInfo_missing    471874
dtype: int64

## 4. Temporal Features

In [12]:
feature_engineer = FeatureEngineer()
transaction_features = feature_engineer.create_transaction_features(feature_base)
transaction_features = feature_engineer.create_missingness_features(transaction_features,missingness_columns)
transaction_features = feature_engineer.create_temporal_features(transaction_features)

In [13]:
transaction_features[["TransactionDT","time_since_previous_transaction","has_previous_transaction"]].head(10)

,TransactionDT,time_since_previous_transaction,has_previous_transaction
0,86400,0.0,0
1,86401,1.0,1
2,86469,68.0,1
3,86499,30.0,1
4,86506,7.0,1
5,86510,4.0,1
6,86522,12.0,1
7,86529,7.0,1
8,86535,6.0,1
9,86536,1.0,1


In [14]:
transaction_features[["time_since_previous_transaction","has_previous_transaction"]].describe()

,time_since_previous_transaction,has_previous_transaction
count,590540.000000,590540.000000
mean,26.627715,0.999998
std,59.299048,0.001301
min,0.000000,0.000000
25%,5.000000,1.000000
50%,13.000000,1.000000
75%,29.000000,1.000000
max,4138.000000,1.000000


## 5. Entity Features

In [15]:
entity_columns = ["card1", "card2", "addr1"]
transaction_features = feature_engineer.create_entity_features(transaction_features,entity_columns)
transaction_features[entity_columns + ["card1_transaction_count","card2_transaction_count","addr1_transaction_count"]].head()

,card1,card2,addr1,card1_transaction_count,card2_transaction_count,addr1_transaction_count
0,13926,NaN,315.0,43,8933,23078
1,2755,404.0,325.0,683,3056,42751
2,4663,490.0,330.0,1108,38145,26287
3,18132,567.0,476.0,4209,6137,9478
4,4497,514.0,420.0,18,14541,3581


In [16]:
transaction_features[["card1_transaction_count","card2_transaction_count","addr1_transaction_count"]].describe()

,card1_transaction_count,card2_transaction_count,addr1_transaction_count
count,590540.000000,590540.000000,590540.000000
mean,2528.815464,18105.872967,27295.782694
std,3702.655513,17746.197502,19938.550411
min,1.000000,14.000000,1.000000
25%,132.000000,2979.000000,8486.000000
50%,919.000000,10126.000000,20827.000000
75%,3152.000000,38145.000000,42751.000000
max,14932.000000,48935.000000,65706.000000


## 6. Card + Device Relationship Features

In [17]:
relationship_features = train_transaction.merge(train_identity[["TransactionID", "DeviceInfo", "DeviceType"]],on="TransactionID",how="left")
transaction_features = feature_engineer.create_relationship_features(
    transaction_features,
    "card1",
    "DeviceInfo"
)

transaction_features[
    ["card1", "DeviceInfo", "card1_DeviceInfo_transaction_count"]
].head()

,card1,DeviceInfo,card1_DeviceInfo_transaction_count
0,13926,NaN,31
1,2755,NaN,621
2,4663,NaN,1099
3,18132,NaN,3916
4,4497,SAMSUNG SM-G892A Build/NRD90M,1


In [18]:
transaction_features[["card1_DeviceInfo_transaction_count"]].describe()

,card1_DeviceInfo_transaction_count
count,590540.000000
mean,1851.580743
std,3321.982860
min,1.000000
25%,51.000000
50%,434.000000
75%,2056.000000
max,14880.000000


In [19]:
relationship_features = train_transaction.merge(train_identity[["TransactionID", "DeviceInfo", "DeviceType"]],on="TransactionID",how="left")
transaction_features = feature_engineer.create_relationship_features(
    transaction_features,
    "card1",
    "DeviceInfo"
)
transaction_features[["card1","DeviceInfo","card1_DeviceInfo_transaction_count"]].head(10)

,card1,DeviceInfo,card1_DeviceInfo_transaction_count
0,13926,NaN,31
1,2755,NaN,621
2,4663,NaN,1099
3,18132,NaN,3916
4,4497,SAMSUNG SM-G892A Build/NRD90M,1
5,5937,NaN,7
6,12308,NaN,203
7,12695,NaN,6589
8,2803,iOS Device,234
9,17399,NaN,1831


In [20]:
transaction_features[["card1_DeviceInfo_transaction_count"]].describe()

,card1_DeviceInfo_transaction_count
count,590540.000000
mean,1851.580743
std,3321.982860
min,1.000000
25%,51.000000
50%,434.000000
75%,2056.000000
max,14880.000000


## 7. Entity-to-Entity Relationship Counts

In [21]:
transaction_features = feature_engineer.create_entity_relationship_count(transaction_features,"card1","DeviceInfo")
transaction_features[["card1","DeviceInfo","card1_unique_DeviceInfo_count"]].head(10)

,card1,DeviceInfo,card1_unique_DeviceInfo_count
0,13926,NaN,4
1,2755,NaN,13
2,4663,NaN,5
3,18132,NaN,34
4,4497,SAMSUNG SM-G892A Build/NRD90M,4
5,5937,NaN,0
6,12308,NaN,2
7,12695,NaN,42
8,2803,iOS Device,59
9,17399,NaN,14


In [22]:
transaction_features[["card1_unique_DeviceInfo_count"]].describe()

,card1_unique_DeviceInfo_count
count,590540.000000
mean,43.993203
std,100.108759
min,0.000000
25%,4.000000
50%,10.000000
75%,37.000000
max,614.000000


## 8. Relationship Insights


 # card1  DeviceInfo Relationship Insight
-----------------------------------------

- The median card1 is associated with 10 unique DeviceInfo values.
- 75% of transactions belong to card1 values associated with 37 or fewer
  unique devices.
- The maximum is 614 unique devices for a card1 value.
- This shows substantial variation in card-to-device relationship breadth.
- The feature can therefore capture entity relationship diversity.
- High relationship diversity is not considered suspicious by itself and
  must be evaluated with fraud behavior later.

In [23]:
transaction_features = feature_engineer.create_entity_relationship_count(transaction_features,"DeviceInfo","card1")

In [24]:
transaction_features[["DeviceInfo", "card1", "DeviceInfo_unique_card1_count"]].head(10)

,DeviceInfo,card1,DeviceInfo_unique_card1_count
0,NaN,13926,0
1,NaN,2755,0
2,NaN,4663,0
3,NaN,18132,0
4,SAMSUNG SM-G892A Build/NRD90M,4497,7
5,NaN,5937,0
6,NaN,12308,0
7,NaN,12695,0
8,iOS Device,2803,3104
9,NaN,17399,0


In [25]:
transaction_features[["DeviceInfo_unique_card1_count"]].describe()

,DeviceInfo_unique_card1_count
count,590540.000000
mean,572.477267
std,1431.815104
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,4846.000000


# Reverse Relationship Insight
----------------------------

- DeviceInfo is highly sparse, which is reflected by the median and
  75th-percentile relationship counts being zero.
- Among transactions with DeviceInfo present, some device values are
  associated with a very large number of unique card1 values.
- The relationship distribution is highly right-skewed, with a maximum
  of 4,846 unique cards associated with a DeviceInfo value.
- Common values such as generic device categories may connect many cards
  and should not automatically be treated as suspicious entities.
- DeviceInfo therefore requires additional analysis to distinguish useful
  specific relationships from common shared descriptors.

## 9. Feature Validation

In [26]:
engineered_features = [
    "log_transaction_amount",
    "addr1_missing",
    "addr2_missing",
    "D7_missing",
    "D12_missing",
    "D13_missing",
    "D14_missing",
    "DeviceInfo_missing",
    "time_since_previous_transaction",
    "has_previous_transaction",
    "card1_transaction_count",
    "card2_transaction_count",
    "addr1_transaction_count",
    "card1_DeviceInfo_transaction_count",
    "card1_unique_DeviceInfo_count",
    "DeviceInfo_unique_card1_count",
]

In [27]:
validator = FeatureValidator()

validation_passed = validator.validate(
    transaction_features,
    engineered_features
)

print("FEATURE VALIDATION PASSED")

FEATURE VALIDATION PASSED
